In [ ]:
import base64
import enum
import json
from typing import Literal
from io import BytesIO
from PIL import Image
import re
import pydantic
import shapely
import geopandas as gpd
from websockets.sync.client import connect
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np

# Client library

In [ ]:
class RadioClimate(enum.Enum):
    EQUATORIAL = 1
    CONTINENTAL_SUBTROPICAL = 2
    MARITIME_TROPICAL = 3
    DESERT = 4
    CONTINENTAL_TEMPERATE = 5
    MARITIME_TEMPERATE_OVER_LAND = 6
    MARITIME_TEMPERATE_OVER_SEA = 7


class Polarization(enum.Enum):
    HORIZONTAL = 0
    VERTICAL = 1


class Radar(pydantic.BaseModel):
    id: int
    lat: float  # [°]
    lon: float  # [°]
    alt: float  # [m]
    power: int  # [W]
    erp: float  # [W] effective radiated power
    antenna_height: float  # [m]
    diameter: float  # antenna diameter [m]
    frequency: float  # [GHz]
    pulse_width: float  # [us]
    cpi_pulses: int
    bandwidth: int  # [MHz]
    pfa: float
    min_elevation: float  # [°]
    max_elevation: float  # [°]
    rotation_time: float  # [s]
    polarization: Polarization


class Target(pydantic.BaseModel):
    lat: float  # [°]
    lon: float  # [°]
    alt: float  # [°]
    cross_section: float  # [m^2] ?
    vlon: float  # [m / s]
    vlat: float  # [m / s]
    vz: float  # [m / s]


class PclReceiver(pydantic.BaseModel):
    name: str
    rx_id: int
    masl: int
    lat: float
    lon: float
    x: int = 0
    y: int = 0
    ahmagl: int = 15
    signal_type: Literal["FM"] = "FM"
    limit_distance: int = 50000
    lostxids: list = []  # IDs of emitters (Tx) for line-of-sight calculation
    status: int = 1
    txcallsigns: str = ""
    bandwidth: int = 8000
    horiz_diagr_att: int = 0
    vert_diagr_att: int = 0
    gain: int = 0
    losses: int = 0
    temp_sys: int = 300
    update_time: int = -1
    txcallsigns: str | None = None
    x: float | None = None
    y: float | None = None


class PclEmitter(pydantic.BaseModel):
    tx_id: int
    callsign: str
    sitename: str
    lat: float
    lon: float
    masl: float
    ahmagl: float
    freq: float
    bandwidth: float
    erp_h: float | Literal["UNDEFINED"]
    erp_v: float | Literal["UNDEFINED"]
    type: Literal["directional", "OMNI"]
    horiz_diagr_att: list[float] | int
    vert_diagr_att: list[float] | Literal["UNDEFINED"]
    pol: Literal["H", "V"]
    signal_type: Literal["FM"]
    losrxids: list[int]
    status: Literal[1]
    power: float


class LatLonHeightGrid(pydantic.BaseModel):
    lat_start: float
    lat_stop: float
    lat_res: float

    lon_start: float
    lon_stop: float
    lon_res: float

    height_start: float
    height_stop: float
    height_res: float

    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)

        # Correct stop values to match the resolution.
        n_points_lat = int(np.ceil((self.lat_stop - self.lat_start) / self.lat_res)) + 1
        n_points_lon = int(np.ceil((self.lon_stop - self.lon_start) / self.lon_res)) + 1
        n_points_height = (
            int(np.ceil((self.height_stop - self.height_start) / self.height_res)) + 1
        )

        self.lat_stop = self.lat_start + (n_points_lat - 1) * self.lat_res
        self.lon_stop = self.lon_start + (n_points_lon - 1) * self.lon_res
        self.height_stop = self.height_start + (n_points_height - 1) * self.height_res

    @property
    def n_points_lat(self) -> int:
        return int(np.round((self.lat_stop - self.lat_start) / self.lat_res)) + 1

    @property
    def n_points_lon(self) -> int:
        return int(np.round((self.lon_stop - self.lon_start) / self.lon_res)) + 1

    @property
    def n_points_height(self) -> int:
        return (
            int(np.round((self.height_stop - self.height_start) / self.height_res)) + 1
        )

    @property
    def n_points(self) -> tuple[int, int, int]:
        return (self.n_points_lat, self.n_points_lon, self.n_points_height)

    def to_openburst(self) -> dict[str, float]:
        return {
            "lat_start": self.lat_start,
            "lat_stop": self.lat_stop,
            "lon_start": self.lon_start,
            "lon_stop": self.lon_stop,
            "min_x": self.lon_start,
            "max_x": self.lon_stop,
            "min_y": self.lat_start,
            "max_y": self.lat_stop,
            "min_z": self.height_start,
            "max_z": self.height_stop,
            "res_x": self.lon_res,
            "res_y": self.lat_res,
            "res_z": self.height_res,
            "amt_pts_x": self.n_points_lon,
            "amt_pts_y": self.n_points_lat,
            "amt_pts_z": self.n_points_height,
        }

    @property
    def center(self) -> tuple[float, float, float]:
        return (
            0.5 * (self.lat_stop + self.lat_start),
            0.5 * (self.lon_stop + self.lon_start),
            0.5 * (self.height_stop + self.height_start),
        )

    @property
    def points(self) -> np.ndarray:
        n = self.n_points[0] * self.n_points[1] * self.n_points[2]
        points = np.empty((n, 3), dtype=np.float32)
        i = 0
        for lat in np.linspace(self.lat_start, self.lat_stop, self.n_points_lat):
            for lon in np.linspace(self.lon_start, self.lon_stop, self.n_points_lon):
                for height in np.linspace(
                    self.height_start, self.height_stop, self.n_points_height
                ):
                    points[i, :] = lat, lon, height
                    i += 1
        return points

In [ ]:
class OpenburstClient:
    _RADAR_COVERAGE: int = 559765
    _ELEVATION: int = 6200901
    _PROPAGATION: int = 29824733

    def __init__(
        self,
        base_url: str,
        radterrain_port: int = 9978,
        geoplot_port: int = 9943,
        pcl_port: int = 7999,
    ):
        self._url_radterrain = f"ws://{base_url}:{radterrain_port}/dem"
        self._url_geoplot = f"ws://{base_url}:{geoplot_port}/geoplot"
        self._url_pcl = f"ws://{base_url}:{pcl_port}/pcl"

    def calculate_coverage(
        self,
        radar: Radar,
        target_flight_height: float,
        target_cross_section: float,
        enable_propagation_model: bool = False,
        enable_magl: bool = False,
    ) -> shapely.geometry.polygon.Polygon:
        """
        Calculate radar coverage.

        Returns
        -------
        Shapely polygon representing radar coverage in the WGS84 coordinate system.
        """
        # Build the request message.
        properties = [
            self._RADAR_COVERAGE,
            radar.id,
            radar.lat,
            radar.lon,
            target_flight_height,
            radar.power,
            radar.diameter,
            radar.frequency / 1000.0,  # convert MHz -> GHz
            radar.pulse_width,
            radar.cpi_pulses,
            radar.bandwidth,
            radar.pfa,
            target_cross_section,
            radar.min_elevation,
            radar.max_elevation,
            int(enable_propagation_model),
            int(enable_magl),
        ]

        # Calculate radar coverage.
        with connect(self._url_radterrain) as ws:
            ws.send(",".join([str(p) for p in properties]))
            answer = ws.recv()

        # Convert radar coverage to KML.
        with connect(self._url_geoplot) as ws:
            ws.send(
                json.dumps(
                    {
                        "request_type": "activeCoveragePoints",
                        "nbr_args": 1,
                        "args": ['"' + answer + '"'],
                    }
                )
            )
            answer2 = ws.recv()
            kml = json.loads(json.loads(answer2)["args"][0])

        # Parse KML file.
        with open("test.kml", "w") as file:
            file.write(kml)

        df = gpd.read_file("test.kml")
        polygon = df.iloc[0]["geometry"]

        return polygon

    def elevation_at(self, query_lat: float, query_lon: float) -> float:
        """Query elevation [m] at the query position."""
        # Needed for distance calculation of query point to POI. Dummy for elevation calc.
        poi_lat = 46.25
        poi_lon = 7.12

        query = [self._ELEVATION, query_lat, query_lon, poi_lat, poi_lon]
        query = [str(part) for part in query]

        with connect(self._url_radterrain) as ws:
            ws.send(",".join(query))
            answer = ws.recv()

        height = json.loads(answer)[1]
        return height

    def get_pcl_receivers_emitters(
        self, receivers: list[PclReceiver], take_non_los_emitters: bool = True
    ) -> tuple[list[PclReceiver], list[PclEmitter]]:
        """
        Query close-by emitters for the given receivers.

        Returns
        -------
        receivers: list[PclReceiver]
            Receivers for the PCL. The property lostxids is set to the IDs of
            the close-by emitters.
        emitters: list[PclSender]
            Suggested emitters to be used for PCL.
        """
        with connect(self._url_pcl) as ws:
            ws.send(
                json.dumps(
                    {
                        "request_type": "findTxForRx_all",
                        "nbr_args": 2,
                        "args": [
                            [r.model_dump_json() for r in receivers],
                            take_non_los_emitters,
                        ],
                    }
                ).encode("utf-8")
            )
            answer = json.loads(ws.recv())

            receivers_obtained = [
                PclReceiver.model_validate_json(config) for config in answer["args"][0]
            ]
            emitters = [
                PclEmitter.model_validate_json(config) for config in answer["args"][1]
            ]
            for recv in receivers_obtained:
                recv.txcallsigns = ",".join([e.callsign for e in emitters])

            return receivers_obtained, emitters

    def calculate_min_det_rcs_coverage(
        self,
        pcl_receivers: list[PclReceiver],
        pcl_emitters: list[PclEmitter],
        grid: LatLonHeightGrid,
        snr_threshold: float = 15.0,
        delay_threshold: float = 1.0,
        integration_time: float = 0.1,
    ) -> list[np.ndarray]:
        with connect(self._url_pcl, ping_timeout=None) as ws:
            ws.send(
                json.dumps(
                    {
                        "request_type": "calcMinDetRCScoverage",
                        "nbr_args": 8,
                        "args": [
                            [r.model_dump_json() for r in pcl_receivers],
                            [e.model_dump_json() for e in pcl_emitters],
                            str(snr_threshold),
                            str(delay_threshold),
                            str(integration_time),
                            json.dumps(grid.to_openburst()),
                            "0",
                            "",
                        ],
                    }
                ).encode("utf-8")
            )
            answer = ws.recv()

        cubes_all_receivers = json.loads(json.loads(answer)["args"][0])
        assert len(cubes_all_receivers) == len(pcl_receivers)
        cubes_all_receivers = [
            np.asarray(json.loads(cube)) for cube in cubes_all_receivers
        ]

        for cube in cubes_all_receivers:
            assert cube.shape == grid.n_points

        return cubes_all_receivers

    def calculate_propagation(
        self,
        transmitter: Radar,
        receiver: Radar,
        radio_climate: RadioClimate = RadioClimate.CONTINENTAL_TEMPERATE,
        earth_dielectric_constant: float = 13.0,  # [no units]
        earth_conductivitiy: float = 0.002,  # [S / m]
        atmospheric_bending_constant: float = 301.0,  # [N-units]
        ground_clutter: float = 3.048,  # [m]; default is 10 feet
        oitm: bool = True,  # use original ITM model
    ):
        METER_TO_FEET = 3.28084
        with connect(self._url_radterrain, ping_timeout=None) as ws:
            params = [
                self._PROPAGATION,
                transmitter.lat,
                -transmitter.lon,
                # convert to feet
                transmitter.antenna_height * METER_TO_FEET,
                receiver.lat,
                -receiver.lon,
                # convert to feet
                receiver.antenna_height * METER_TO_FEET,
                earth_dielectric_constant,
                earth_conductivitiy,
                atmospheric_bending_constant,
                transmitter.frequency,
                radio_climate.value,
                transmitter.polarization.value,
                ground_clutter,
                int(oitm),
                transmitter.erp,
            ]
            ws.send(",".join([str(p) for p in params]))
        answer = ws.recv()

        # Load the image.
        img_str = json.loads(answer)[1]["height_profile.png"]
        image_bytes = base64.b64decode(img_str)
        img = Image.open(BytesIO(image_bytes))

        # Extract the losses.
        text = json.loads(answer)[1]["TX_-to-RX_.txt"]

        free_space_loss_match = re.search(
            r"Free space path loss:\s*([0-9]+(?:\.[0-9]+)?)\s*dB", text
        )
        longley_rice_loss_match = re.search(
            r"Longley-Rice path loss:\s*([0-9]+(?:\.[0-9]+)?)\s*dB", text
        )
        terrain_shielding_loss_match = re.search(
            r"Attenuation due to terrain shielding:\s*([0-9]+(?:\.[0-9]+)?)\s*dB", text
        )

        free_space_loss = (
            free_space_loss_match.group(1) if free_space_loss_match else np.nan
        )
        longley_rice_loss = (
            longley_rice_loss_match.group(1) if longley_rice_loss_match else np.nan
        )
        terrain_shielding_loss = (
            terrain_shielding_loss_match.group(1)
            if terrain_shielding_loss_match
            else np.nan
        )

        return img, free_space_loss, longley_rice_loss, terrain_shielding_loss


In [ ]:
def coverage_polygon_to_mask(
    polygon: shapely.geometry.polygon.Polygon,
    center_lon: float,
    center_lat: float,
    extent_lon: float,
    extent_lat: float,
    width: int,
    height: int,
) -> np.ndarray:
    # Calculate region represented by the mask.
    lon_min = center_lon - 0.5 * extent_lon
    lon_max = center_lon + 0.5 * extent_lon

    lat_min = center_lat - 0.5 * extent_lat
    lat_max = center_lat + 0.5 * extent_lat

    # Clip overlay to bbox.
    bbox = gpd.GeoDataFrame(
        geometry=[
            gpd.points_from_xy([lon_min, lon_max], [lat_min, lat_max])
            .union_all()
            .envelope
        ],
        crs="EPSG:4326",
    )
    df = gpd.GeoDataFrame([], geometry=[polygon])
    df.set_crs("EPSG:4326", inplace=True)
    overlay_clipped = df.clip(bbox)

    # Raster transform.
    transform = from_bounds(lon_min, lat_min, lon_max, lat_max, width, height)

    # Rasterize to create binary mask.
    mask = rasterize(
        [(geom, 1) for geom in overlay_clipped.geometry],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype=np.uint8,
    )
    return mask

# Config

In [ ]:
class Radar(pydantic.BaseModel):
    id: int
    lat: float  # [°]
    lon: float  # [°]
    alt: float  # [m]
    power: int  # [W]
    erp: float  # [W] effective radiated power
    antenna_height: float  # [m]
    diameter: float  # antenna diameter [m]
    frequency: float  # [GHz]
    pulse_width: float  # [us]
    cpi_pulses: int
    bandwidth: int  # [MHz]
    pfa: float
    min_elevation: float  # [°]
    max_elevation: float  # [°]
    rotation_time: float  # [s]
    polarization: Polarization


transmitter = Radar(
    id=585,
    lat=47.36700085728634,
    lon=8.537724304199216,
    alt=407.83600886023686,
    power=20000,
    erp=800,
    antenna_height=10.0,
    diameter=2.0,
    frequency=1000.0,
    pulse_width=1,
    cpi_pulses=1,
    bandwidth=1,
    pfa=1e-6,
    min_elevation=-20.0,
    max_elevation=60.0,
    rotation_time=10.0,
    polarization=Polarization.HORIZONTAL,
)

receiver = transmitter.model_copy()
receiver.lat = 47.28968537077668
receiver.lon = 8.608019638061526
receiver.alt = 160.6

target_flight_height: float = 1000.0
target_cross_section: float = 2.0

# Parameters for controlling the resolution of the binary mask.
# Mask extent in degrees.
extent = 0.6

# Mask resolution in pixels.
width, height = 1024, 1024

# Run

In [ ]:
client = OpenburstClient("localhost")
result = client.calculate_propagation(transmitter, receiver)
result

In [ ]:
%%timeit
result = client.calculate_propagation(transmitter, receiver)

In [ ]:
client = OpenburstClient("localhost")
coverage = client.calculate_coverage(
    transmitter,
    target_flight_height,
    target_cross_section,
)

In [ ]:
lon, lat = coverage.exterior.coords.xy
with open("reference.geosjon", "w") as file:
    json.dump(
        {
            "type": "Polygon",
            "coordinates": [np.stack([lon, lat], axis=1).tolist()],
        },
        file,
    )

In [ ]:
with open("reference.geojson", "w") as file:
    json.dump(json.loads(gpd.GeoSeries([coverage]).to_json()), file)

# Plot

In [ ]:
import folium

In [ ]:
m = folium.Map(location=(46.7, 10.3), zoom_start=8)
overlay = folium.GeoJson(coverage).add_to(m)

m.save("map.html")

# Rendering to image

In [ ]:
mask = coverage_polygon_to_mask(
    coverage, transmitter.lon, transmitter.lat, extent, extent, width, height
)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(mask, aspect=1.0)

# Query elevation

In [ ]:
client = OpenburstClient("localhost")
client.elevation_at(47.38301680962604, 8.495104310332978)

In [ ]:
coverage

# PCL

In [ ]:
receivers = [
    PclReceiver(
        name="PCL_SENSOR_0",
        rx_id=1,
        lat=46.95556964077278,
        lon=8.638317871093756,
        masl=1151,
    ),
    PclReceiver(
        name="PCL_SENSOR_1",
        rx_id=2,
        lat=46.99679762141241,
        lon=8.881390380859381,
        masl=1844,
    ),
]

In [ ]:
client = OpenburstClient("localhost")
# Get "Emitters of opportunity".
pcl_receivers, pcl_emitters = client.get_pcl_receivers_emitters(receivers)

In [ ]:
import pandas as pd

In [ ]:
pd.DataFrame([r.model_dump() for r in pcl_receivers]).set_index("rx_id")

In [ ]:
pd.DataFrame([e.model_dump() for e in pcl_emitters]).set_index("tx_id")

In [ ]:
client = OpenburstClient("localhost")
# Get "Emitters of opportunity".
pcl_receivers, pcl_emitters = client.get_pcl_receivers_emitters(receivers)

grid = LatLonHeightGrid(
    lat_start=46.85611714645961,
    lat_stop=46.99679762141241,
    lat_res=0.015631163883644678,
    lon_start=8.300488281249999,
    lon_stop=8.881390380859381,
    lon_res=0.06454467773437579,
    height_start=3000.0,
    height_stop=3000.0,
    height_res=3000.0,
)

In [ ]:
cubes_per_receiver = client.calculate_min_det_rcs_coverage(
    pcl_receivers, pcl_emitters, grid
)

In [ ]:
with open("output.txt", "r") as file:
    lines = file.readlines()
lines = [l for l in lines if l.startswith("Time for")]
times = [float(l.split(":")[1].strip().replace("s", "")) for l in lines]
sum(times)

In [ ]:
transmitter.model_dump_json()

In [ ]:
import requests


def rad_detection(
    radar: Radar, target: Target, doppler_shift_threshold: float = 5.0
) -> float:
    response = requests.get(
        "http://localhost:3875/rad_detection",
        params={
            "radar": radar.model_dump_json(),
            "target": target.model_dump_json(),
            "doppler_shift_threshold": doppler_shift_threshold,
        },
    )

    return float(response.text)

In [ ]:
target = Target(
    lat=47.348,
    lon=8.6266,
    alt=1000.0,
    cross_section=1.0,
    vlon=250.0,
    vlat=0.0,
    vz=0.0,
)

In [ ]:
rad_detection(transmitter, target)

In [ ]:
import matplotlib.pyplot as plt
import geojsoncontour

fig, ax = plt.subplots()

data = cubes_per_receiver[0]

contour = ax.contourf(
    np.linspace(grid.lon_start, grid.lon_stop, grid.n_points_lon),
    np.linspace(grid.lat_start, grid.lat_stop, grid.n_points_lat),
    data[:, :, 0],
)
geojson = geojsoncontour.contourf_to_geojson(contourf=contour, ndigits=3, unit="m")
plt.close(fig)

import folium
from folium.plugins import HeatMap

m = folium.Map(location=(grid.center[0], grid.center[1]))
for receiver in pcl_receivers:
    folium.Marker(
        location=(receiver.lat, receiver.lon),
        tooltip=receiver.name,
        icon=folium.Icon(icon="ear-listen", prefix="fa", color="blue"),
    ).add_to(m)
for emitter in pcl_emitters:
    folium.Marker(
        location=(emitter.lat, emitter.lon),
        tooltip=emitter.callsign,
        icon=folium.Icon(icon="tower-cell", prefix="fa", color="blue"),
    ).add_to(m)
folium.GeoJson(geojson).add_to(m)
m.save("map.html")

In [ ]:
polygon = contour.get_paths()[0].to_polygons()[0]  # [[0, -1]]

In [ ]:
type(contour.get_paths()[0])

In [ ]:
import matplotlib

def polygon_to_geojson(polygon: np.ndarray) -> str:
    """
    Parameters
    ----------
    polygon: np.ndarray
        Array of shape (N, 2) containing (lon, lat) coordinates as rows.
    """
    return {
        "type": "feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": polygon.tolist(),
        }
    }


def path_to_geojson(path: matplotlib.path.Path) -> str:


polygon_to_geojson(polygon)

# Load replay data

In [ ]:
import pandas as pd

cols = [
    "DateTimeIndex",
    "millisecs",
    "converted_integer_id",
    "lat",
    "lon",
    "heading[0 = north\n180 = south\n360 = north]",
    "speed [km / h]",
    "altitude[m]",
    "track_quality",
    "milli_secs_after_midnight",
    "tgt_vx [vel m/s on lon axis]",
    "tgt_vy [vel m/s on lat axis]",
    "tgt_vz [vel m/s on z axis]",
]

df = pd.DataFrame(
    np.load(
        "./openburst/replay/DATA/eastbound_single_4000masl.npy",
        allow_pickle=True,
    ),
    columns=cols,
)

# Only one target.
assert len(df["converted_integer_id"].unique()) == 1

path = {
    "type": "LineString",
    "coordinates": df[["lon", "lat"]].values.tolist(),
}

In [ ]:
map = folium.Map()

folium.GeoJson(path).add_to(map)
map.save("map.html")

In [ ]:
delta_t = df["DateTimeIndex"] - df["DateTimeIndex"].iloc[0]

In [ ]:
delta_t.dt.microseconds

In [ ]:
df